# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Data loading

In [1]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(9841378, 31)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The selected method is Random Forest classification.

The purpose of this lane is to identify content items that may require refresh attention. A tree-based model was selected because it can capture non-linear relationships between ranking and engagement signals without requiring strong assumptions about the data distribution.

The model uses independent signals that were not directly used to create the Week-4 baseline rule:

- gsc_avg_position
- ga4_pageviews
- ga4_users
- ga4_total_engagement_sec
- scroll_events

The goal is not only to improve the baseline score, but to test whether additional behavioural and ranking signals can identify refresh opportunities beyond the existing rule-based approach.

A Random Forest was chosen because it provides both predictive capability and feature importance interpretation, allowing the model decisions to be inspected.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score

print("Libraries loaded")

Libraries loaded


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped split.

Content from the same client should not appear in both training and testing because that would allow the model to memorize client-specific patterns.

A grouped split better represents performance on unseen clients.

#### Define modelling features and create proxy label

In [8]:
features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_users",
    "ga4_total_engagement_sec",
    "scroll_events"
]


model_df = df.copy()


# create a realistic proxy label
model_df["refresh_label"] = (
    (model_df["gsc_impressions"] > 100)
    &
    (model_df["gsc_clicks"] <= 1)
    &
    (model_df["ga4_sessions"].fillna(0) == 0)
).astype(int)


print(model_df["refresh_label"].value_counts())

refresh_label
0    9452885
1     388493
Name: count, dtype: int64


#### Perform client-grouped train-test split

In [9]:
X = model_df[features].fillna(0)

y = model_df["refresh_label"]

groups = model_df["client_hash_id"]


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(X,y,groups)
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (8935676, 5)
Test: (905702, 5)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest model was trained and evaluated using the same client-grouped split as the Week-4 baseline comparison.

The evaluation uses the same classification metrics:

- Precision
- Recall
- F1-score

The baseline performs better because it directly uses the strongest search performance signals (`gsc_impressions` and `gsc_clicks`) that define the refresh opportunity rule.

The Random Forest model uses indirect signals and therefore attempts to approximate the baseline decision using different information.

The comparison shows that the baseline remains stronger for this specific proxy definition, while the model provides additional insight into which independent signals are associated with refresh opportunities.

In [10]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)


model.fit(
    X_train,
    y_train
)


model_predictions = model.predict(X_test)


print("Model trained")

Model trained


## Baseline

In [14]:
# Build the baseline on the original dataframe
baseline_test = model_df.iloc[test_idx]

baseline_predictions = (
    (baseline_test["gsc_impressions"] > 100)
    &
    (baseline_test["gsc_clicks"] <= 1)
).astype(int)

## Comparison

In [16]:
comparison = pd.DataFrame({
    "Approach": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision": [
        precision_score(y_test, baseline_predictions),
        precision_score(y_test, model_predictions)
    ],
    "Recall": [
        recall_score(y_test, baseline_predictions),
        recall_score(y_test, model_predictions)
    ],
    "F1": [
        f1_score(y_test, baseline_predictions),
        f1_score(y_test, model_predictions)
    ]
})

comparison

,Approach,Precision,Recall,F1
0,Week-4 Baseline,0.767525,1.000000,0.868474
1,Random Forest,0.309382,0.692788,0.427744


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest model was evaluated through feature importance analysis and error inspection.

The model relied mainly on `gsc_avg_position`, which contributed approximately 96% of the feature importance. This indicates that ranking position was the strongest independent signal available for identifying potential refresh opportunities.

The remaining behavioural features (`ga4_users`, `ga4_pageviews`, `ga4_total_engagement_sec`, and `scroll_events`) contributed relatively little. This is likely because many records contain limited GA4 behavioural information.

False positives were mainly pages that the model identified as potential refresh candidates but were not included in the proxy label. These cases represent situations where ranking-related signals suggested an opportunity, but the baseline conditions were not satisfied.

False negatives were pages that matched the refresh proxy label but were missed by the model. These cases show that pages can have acceptable ranking positions while still experiencing poor click performance.

Overall, the Random Forest model did not outperform the Week-4 baseline because the baseline directly uses the strongest signals defining the proxy label. However, the model provided useful directional insights by showing which independent signals contributed to refresh prediction and where prediction errors occurred.

The result demonstrates that model complexity alone does not guarantee improvement over a well-designed rule-based baseline.

In [17]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
0,gsc_avg_position,0.962400
2,ga4_users,0.023190
1,ga4_pageviews,0.012988
3,ga4_total_engagement_sec,0.001336
4,scroll_events,0.000087


## Errors

In [18]:
errors = X_test.copy()

errors["actual"] = y_test.values
errors["predicted"] = model_predictions

false_positive = errors[
    (errors.actual == 0) &
    (errors.predicted == 1)
]

false_negative = errors[
    (errors.actual == 1) &
    (errors.predicted == 0)
]

print("False positives:")
display(false_positive.head(3))

print("False negatives:")
display(false_negative.head(3))

False positives:


,gsc_avg_position,ga4_pageviews,ga4_users,ga4_total_engagement_sec,scroll_events,actual,predicted
18876,14.078947,0.0,0.0,0.0,0.0,0,1
18884,3.541667,0.0,0.0,0.0,0.0,0,1
18893,5.461538,0.0,0.0,0.0,0.0,0,1


False negatives:


,gsc_avg_position,ga4_pageviews,ga4_users,ga4_total_engagement_sec,scroll_events,actual,predicted
19047,9.952381,0.0,0.0,0.0,0.0,1,0
19049,8.439024,0.0,0.0,0.0,0.0,1,0
19107,8.306122,0.0,0.0,0.0,0.0,1,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.